# Stochastic gradient descent

Diabetes data, 10 inputs. One random row per update.

In [ ]:
import time, warnings
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
print("OLS test R2", LinearRegression().fit(X_train, y_train).score(X_test, y_test))

## Our own stochastic gradient descent

In [ ]:
class MySGDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100, seed=0):
        self.lr, self.epochs = learning_rate, epochs
        self.rng = np.random.default_rng(seed)       # fixed seed: same random rows every run

    def fit(self, X, y):
        n = X.shape[0]
        self.intercept_ = 0.0
        self.coef_ = np.ones(X.shape[1])
        for _ in range(self.epochs):
            for _ in range(n):                        # n updates per epoch
                i = self.rng.integers(n)              # one random row
                err = y[i] - (X[i] @ self.coef_ + self.intercept_)
                self.intercept_ -= self.lr * (-2 * err)
                self.coef_ -= self.lr * (-2 * err * X[i])
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

start = time.perf_counter()
sgd = MySGDRegressor(learning_rate=0.01, epochs=40).fit(X_train, y_train)
print("seconds", round(time.perf_counter() - start, 3), "test R2", r2_score(y_test, sgd.predict(X_test)))

## Same number of epochs, batch gradient descent

In [ ]:
b, w = 0.0, np.ones(10)
for _ in range(40):
    err = y_train - (X_train @ w + b)
    b -= 0.5 * (-2 * err.mean())
    w -= 0.5 * (-2 * X_train.T @ err / len(X_train))
print("batch, 40 epochs, test R2", r2_score(y_test, X_test @ w + b))

## A learning schedule: the rate shrinks as training goes on

In [ ]:
t0, t1 = 5, 50
schedule = lambda t: t0 / (t + t1)
print([round(schedule(t), 4) for t in [0, 10, 100, 1000, 10000]])

## scikit-learn's SGDRegressor

In [ ]:
warnings.simplefilter("ignore")      # it warns when max_iter is reached before its stopping rule
for rate in ["constant", "invscaling"]:
    reg = SGDRegressor(max_iter=100, learning_rate=rate, eta0=0.01, random_state=0).fit(X_train, y_train)
    print(rate, "test R2", round(reg.score(X_test, y_test), 4), "epochs run", reg.n_iter_)